In [ ]:
import os
import sys
import warnings
sys.path.append(os.path.abspath(os.path.join("..")))

from configs.config import BRONZE_DATA_DIR, SILVER_DATA_DIR
from src.spark import get_spark_session

warnings.filterwarnings("ignore")

# `READ DATA FROM BRONZE`

In [ ]:
spark = get_spark_session()

spark

In [ ]:
bronze_path = BRONZE_DATA_DIR / "youtube_channels"

In [ ]:
df_silver = spark.read.parquet(str(bronze_path))

In [ ]:
df_silver.show()

+----+--------------------+-----------+--------------+-----------+--------------------+-------+
|rank|            Youtuber|subscribers|   video views|video count|            category|started|
+----+--------------------+-----------+--------------+-----------+--------------------+-------+
|   1|        Tsuriki Show| 34,100,000|42,490,526,838|      4,739|       Entertainment|   2019|
|   2|Kidibli (Kinder S...| 29,600,000|15,673,364,837|      1,236|       Entertainment|   2015|
|   3|Kurzgesagt – In a...| 23,600,000| 3,145,706,013|        271|           Education|   2013|
|   4|            boxtoxtv| 23,500,000|18,303,986,629|      1,559|              Comedy|   2022|
|   5|          HaerteTest| 19,500,000| 3,420,864,412|      1,712|Science & Technology|   2011|
|   6|       Noel Robinson| 18,600,000|10,752,582,783|      1,639|       Entertainment|   2015|
|   7|        FAMILY BOOMS| 16,800,000|16,225,259,066|      1,587|       Entertainment|   2021|
|   8|      Talking Angela| 13,300,000| 

In [ ]:
df_silver.printSchema()

root
 |-- rank: integer (nullable = true)
 |-- Youtuber: string (nullable = true)
 |-- subscribers: string (nullable = true)
 |-- video views: string (nullable = true)
 |-- video count: string (nullable = true)
 |-- category: string (nullable = true)
 |-- started: integer (nullable = true)



# `DESCRIBE TABLE BEFORE TRANSFORMATION`

In [ ]:
df_silver.summary().show()

+-------+-----------------+--------------------+-----------+-----------+------------------+--------------------+------------------+
|summary|             rank|            Youtuber|subscribers|video views|       video count|            category|           started|
+-------+-----------------+--------------------+-----------+-----------+------------------+--------------------+------------------+
|  count|             1000|                1000|       1000|       1000|              1000|                1000|              1000|
|   mean|            500.5|                NULL|       NULL|        0.0|429.75042735042734|                NULL|          2014.745|
| stddev|288.8194360957494|                NULL|       NULL|        0.0|274.27472092474073|                NULL|4.2435813863979615|
|    min|                1|   #Mentale Zuflucht|  1,000,000|          0|                 0|    Autos & Vehicles|              2005|
|    25%|              250|                NULL|       NULL|        0.0|    

### `TRANSFORMATION 1: CLEAN COLUMN NAMES`

In [ ]:
df_silver = df_silver.withColumnRenamed("video views", "video_views")
df_silver = df_silver.withColumnRenamed("video count", "video_count")
df_silver = df_silver.withColumnRenamed("Youtuber", "youtuber")

### `TRANSFORMATION 2: TRIMMING WHITESPACES`

In [ ]:
from pyspark.sql import functions
col_trim = ["youtuber", "category"]
for col in col_trim:
    df_silver = df_silver.withColumn(
        col,
        functions.initcap(functions.trim(col))
    )

### `TRANSFORMATION 3: DEDUPLICATION`

In [ ]:
duplicate_count = (
    df_silver.count() - df_silver.drop_duplicates().count()
)

print(f"Duplicate_rows : {duplicate_count}")

Duplicate_rows : 0


In [ ]:
df_silver.groupBy("rank").count().filter(functions.col("count") > 1).show()

+----+-----+
|rank|count|
+----+-----+
+----+-----+



### `TRANSFORMATION 4: NULL CHECKS`

In [ ]:
null_checks = df_silver.select([
    functions.sum(functions.col(c).isNull().cast("int")).alias(c)
    for c in df_silver.columns
])

null_checks.show()

+----+--------+-----------+-----------+-----------+--------+-------+
|rank|youtuber|subscribers|video_views|video_count|category|started|
+----+--------+-----------+-----------+-----------+--------+-------+
|   0|       0|          0|          0|          0|       0|      0|
+----+--------+-----------+-----------+-----------+--------+-------+



### `TRANSFORMATION 5: TYPE CASTING`

In [ ]:
from pyspark.sql.functions import col, regexp_replace
col_int = ["subscribers", "video_views", "video_count"]
for col_name in col_int:
    df_silver = df_silver.withColumn(
        col_name,
        regexp_replace(col(col_name), ",","").cast("long")
    )

In [ ]:
df_silver.show()

+----+--------------------+-----------+-----------+-----------+--------------------+-------+
|rank|            youtuber|subscribers|video_views|video_count|            category|started|
+----+--------------------+-----------+-----------+-----------+--------------------+-------+
|   1|        Tsuriki Show|   34100000|42490526838|       4739|       Entertainment|   2019|
|   2|Kidibli (kinder S...|   29600000|15673364837|       1236|       Entertainment|   2015|
|   3|Kurzgesagt – In A...|   23600000| 3145706013|        271|           Education|   2013|
|   4|            Boxtoxtv|   23500000|18303986629|       1559|              Comedy|   2022|
|   5|          Haertetest|   19500000| 3420864412|       1712|Science & Technology|   2011|
|   6|       Noel Robinson|   18600000|10752582783|       1639|       Entertainment|   2015|
|   7|        Family Booms|   16800000|16225259066|       1587|       Entertainment|   2021|
|   8|      Talking Angela|   13300000| 4116834514|        345|       

### `TRANSFORMATION 6: RANGE VALIDATION`

In [ ]:
col_range = ["rank", "video_count", "video_views"]
for col_name in col_range:
    invalid_ranges = df_silver.filter(functions.col(col_name) <= 0).count()
    print(f"invalid {col_name} records: {invalid_ranges}")

invalid rank records: 0
invalid video_count records: 8
invalid video_views records: 7


In [ ]:
df_silver.select("*").filter("video_count <= 0").show()

+----+--------------------+-----------+-----------+-----------+----------------+-------+
|rank|            youtuber|subscribers|video_views|video_count|        category|started|
+----+--------------------+-----------+-----------+-----------+----------------+-------+
| 147|يوسف الصدّيق / Pr...|    2320000|          0|          0|Film & Animation|   2018|
| 160|          Yalla Gang|    2210000|          0|          0|   Entertainment|   2011|
| 246|         Spacex Inc.|    1710000|   20571411|          0|           Music|   2011|
| 310|            Lpmitkev|    1480000|          0|          0|          Gaming|   2011|
| 435|               Taddl|    1160000|          0|          0|          Gaming|   2009|
| 454|                Unge|    1120000|          0|          0|   Entertainment|   2014|
| 668|          مارد العرب|     803000|          0|          0|       Education|   2017|
| 975|           Mainstage|     570128|          0|          0|   Entertainment|   2012|
+----+---------------

In [ ]:
df_silver.select("*").filter("video_views <= 0").show()

+----+--------------------+-----------+-----------+-----------+----------------+-------+
|rank|            youtuber|subscribers|video_views|video_count|        category|started|
+----+--------------------+-----------+-----------+-----------+----------------+-------+
| 147|يوسف الصدّيق / Pr...|    2320000|          0|          0|Film & Animation|   2018|
| 160|          Yalla Gang|    2210000|          0|          0|   Entertainment|   2011|
| 310|            Lpmitkev|    1480000|          0|          0|          Gaming|   2011|
| 435|               Taddl|    1160000|          0|          0|          Gaming|   2009|
| 454|                Unge|    1120000|          0|          0|   Entertainment|   2014|
| 668|          مارد العرب|     803000|          0|          0|       Education|   2017|
| 975|           Mainstage|     570128|          0|          0|   Entertainment|   2012|
+----+--------------------+-----------+-----------+-----------+----------------+-------+



In [ ]:
df_silver = df_silver.filter((df_silver["video_count"] > 0) & (df_silver["video_views"] > 0))

In [ ]:
df_silver.show()

+----+--------------------+-----------+-----------+-----------+--------------------+-------+
|rank|            youtuber|subscribers|video_views|video_count|            category|started|
+----+--------------------+-----------+-----------+-----------+--------------------+-------+
|   1|        Tsuriki Show|   34100000|42490526838|       4739|       Entertainment|   2019|
|   2|Kidibli (kinder S...|   29600000|15673364837|       1236|       Entertainment|   2015|
|   3|Kurzgesagt – In A...|   23600000| 3145706013|        271|           Education|   2013|
|   4|            Boxtoxtv|   23500000|18303986629|       1559|              Comedy|   2022|
|   5|          Haertetest|   19500000| 3420864412|       1712|Science & Technology|   2011|
|   6|       Noel Robinson|   18600000|10752582783|       1639|       Entertainment|   2015|
|   7|        Family Booms|   16800000|16225259066|       1587|       Entertainment|   2021|
|   8|      Talking Angela|   13300000| 4116834514|        345|       

In [ ]:
df_silver.select("*").filter("video_views <= 0").show()

+----+--------+-----------+-----------+-----------+--------+-------+
|rank|youtuber|subscribers|video_views|video_count|category|started|
+----+--------+-----------+-----------+-----------+--------+-------+
+----+--------+-----------+-----------+-----------+--------+-------+



In [ ]:
df_silver = df_silver.withColumn(
    "views_per_subscriber",
    functions.when(
    functions.col("subscribers") > 0, 
    functions.col("video_views") / functions.col("subscribers")
    )
)

In [ ]:
df_silver = df_silver.withColumn(
    "avg_views_per_video",
    functions.when(
    functions.col("video_count") > 0, 
    functions.col("video_views") / functions.col("video_count")
    )
)

In [ ]:
df_silver.show()

+----+--------------------+-----------+-----------+-----------+--------------------+-------+--------------------+--------------------+
|rank|            youtuber|subscribers|video_views|video_count|            category|started|views_per_subscriber| avg_views_per_video|
+----+--------------------+-----------+-----------+-----------+--------------------+-------+--------------------+--------------------+
|   1|        Tsuriki Show|   34100000|42490526838|       4739|       Entertainment|   2019|  1246.0565055131965|    8966137.75859886|
|   2|Kidibli (kinder S...|   29600000|15673364837|       1236|       Entertainment|   2015|   529.5055688175676|1.2680715887540452E7|
|   3|Kurzgesagt – In A...|   23600000| 3145706013|        271|           Education|   2013|  133.29262766949154|1.1607771265682656E7|
|   4|            Boxtoxtv|   23500000|18303986629|       1559|              Comedy|   2022|   778.8930480425532|1.1740850948685054E7|
|   5|          Haertetest|   19500000| 3420864412|    

# `WRITE TO SILVER`

In [ ]:
silver_path = SILVER_DATA_DIR / "youtube_channels"

(
    df_silver.write.mode("overwrite").parquet(str(silver_path))
)

In [ ]:
silver_check = spark.read.parquet(silver_path)

In [ ]:
silver_check.show(10)

In [ ]:
silver_check.printSchema()

In [ ]:
print(f"Slver rows: {silver_check.count()}")